# VIX Futures Roll Trading Strategy Backtest

**Based on:**
- Buehler & Cusatis (2018) — *Trading the VIX Futures Roll Using Exchange-Traded Funds*
- Tony Cooper (2013) — *Easy Volatility Investing*

This notebook:
1. Loads VIX futures data (CBOE historical CSVs)
2. Downloads ETF prices (SVXY, VIXY, SPY) and VIX spot via `yfinance`
3. Constructs the daily futures term structure and computes the **daily roll**
4. Implements the Buehler trading strategy with optimised contango/backwardation thresholds
5. Also implements Cooper's Vratio (Strategy 3) and VRP (Strategy 4) for comparison
6. Produces equity-curve charts matching Buehler Exhibit 6

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os, glob, re, warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Install yfinance if needed
try:
    import yfinance as yf
except ImportError:
    !pip install yfinance
    import yfinance as yf

print('All imports OK')

## 2. Configuration

Point `VIX_FUTURES_DIR` to the folder where your CBOE VIX futures CSVs live.

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
VIX_FUTURES_DIR = 'data/vix_futures'   # <-- adjust to your path

# Buehler strategy thresholds (from their test-period optimisation)
SHORT_THRESHOLD =  0.0197   # daily roll >= this → sell vol (buy SVXY)
LONG_THRESHOLD  = -0.2400   # daily roll <= this → buy vol  (buy VIXY)

# Date ranges
TEST_START  = '2008-01-02'
TEST_END    = '2010-12-31'
OOS_START   = '2011-10-04'  # SVXY inception
OOS_END     = '2016-12-30'
FULL_END    = '2025-12-31'  # extend as far as data allows

# Month-code mapping for old-format CBOE files
MONTH_CODES = {
    'F': 1, 'G': 2, 'H': 3, 'J': 4, 'K': 5, 'M': 6,
    'N': 7, 'Q': 8, 'U': 9, 'V': 10, 'X': 11, 'Z': 12
}
print('Config loaded.')

## 3. Load VIX Futures Data

We parse both the **old archive format** (`CFE_F08_VX.csv`) and the **new format** (`VX_2015-07-22.csv`).
From each file we extract the **settlement date** (= expiration) and daily settle prices.

In [ ]:
def parse_expiration_from_filename(fname):
    """
    Old format:  CFE_F08_VX.csv  → month-code F, year 08 → Jan 2008
    New format:  VX_2015-07-22.csv → expiration date 2015-07-22
    Returns the expiration date (or approximate month-end for old format).
    """
    base = os.path.basename(fname)
    
    # New format: VX_YYYY-MM-DD.csv
    m = re.match(r'VX_(\d{4}-\d{2}-\d{2})\.csv', base)
    if m:
        return pd.Timestamp(m.group(1))
    
    # Old format: CFE_X09_VX.csv  (month-code + 2-digit year)
    m = re.match(r'CFE_([A-Z])(\d{2})_VX\.csv', base)
    if m:
        month_code, yy = m.group(1), int(m.group(2))
        year = 2000 + yy
        month = MONTH_CODES.get(month_code)
        if month is None:
            return None
        # VIX futures expire on the Wednesday 30 days before
        # the 3rd Friday of the following month. We approximate
        # with the 3rd Wednesday of the expiration month.
        first_day = pd.Timestamp(year, month, 1)
        # Find 3rd Wednesday: first Wednesday + 14 days
        wed_offset = (2 - first_day.weekday()) % 7  # days to first Wednesday
        third_wed = first_day + pd.Timedelta(days=wed_offset + 14)
        return third_wed
    
    return None


def load_single_contract(fpath):
    """Load one VIX futures CSV, return cleaned DataFrame."""
    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        return None
    
    # Standardize column names
    df.columns = df.columns.str.strip()
    
    # Parse trade date
    if df['Trade Date'].str.contains('/').any():
        df['Trade Date'] = pd.to_datetime(df['Trade Date'], format='%m/%d/%Y')
    else:
        df['Trade Date'] = pd.to_datetime(df['Trade Date'])
    
    df = df.rename(columns={'Trade Date': 'date'})
    
    # Get expiration from filename
    expiry = parse_expiration_from_filename(fpath)
    if expiry is None:
        return None
    df['expiry'] = expiry
    
    # Use Settle price, but fall back to Close when Settle is 0
    # (some early new-format files 2013 have Settle=0 with prices in Close)
    df['Settle'] = pd.to_numeric(df['Settle'], errors='coerce')
    df['Close']  = pd.to_numeric(df['Close'], errors='coerce')
    df['Settle'] = df['Settle'].where(df['Settle'] > 0, df['Close'])
    df = df[df['Settle'] > 0].copy()
    
    return df[['date', 'expiry', 'Settle', 'Total Volume', 'Open Interest']].copy()


# Load all contracts
all_files = sorted(glob.glob(os.path.join(VIX_FUTURES_DIR, '*.csv')))
print(f'Found {len(all_files)} VIX futures CSV files')

frames = []
for f in all_files:
    df = load_single_contract(f)
    if df is not None and len(df) > 0:
        frames.append(df)

futures_all = pd.concat(frames, ignore_index=True)
futures_all = futures_all.sort_values(['date', 'expiry']).reset_index(drop=True)

# ── Drop pre-2008 data (old archive format has quality issues) ──
futures_all = futures_all[futures_all['date'] >= '2008-01-01'].reset_index(drop=True)

print(f'Total rows: {len(futures_all):,}')
print(f'Date range: {futures_all.date.min().date()} to {futures_all.date.max().date()}')
print(f'Unique contracts: {futures_all.expiry.nunique()}')
futures_all.head()

## 4. Build Daily Term Structure

For each trading day, identify the **nearest** and **next** futures contracts and compute
the **daily roll** in VIX futures points (Buehler's metric):

$$\text{daily roll} = \frac{\text{VIX}_{\text{spot}} - F_{\text{nearest}}}{\text{trading days to settlement}}$$

In [ ]:
def build_term_structure(futures_all):
    """
    For each trading day, find the nearest and next contracts.
    Returns a DataFrame indexed by date with columns:
      F1_settle, F1_expiry, F1_dte (days-to-expiry),
      F2_settle, F2_expiry, F2_dte
    """
    # Only keep contracts where expiry > trade date (not yet expired)
    df = futures_all[futures_all['expiry'] > futures_all['date']].copy()
    df['dte'] = (df['expiry'] - df['date']).dt.days
    
    records = []
    for date, group in df.groupby('date'):
        # Sort by expiry to get nearest first
        g = group.sort_values('expiry')
        
        if len(g) < 2:
            continue
        
        f1 = g.iloc[0]
        f2 = g.iloc[1]
        
        records.append({
            'date': date,
            'F1_settle': f1['Settle'],
            'F1_expiry': f1['expiry'],
            'F1_dte': f1['dte'],
            'F2_settle': f2['Settle'],
            'F2_expiry': f2['expiry'],
            'F2_dte': f2['dte'],
        })
    
    ts = pd.DataFrame(records)
    ts['date'] = pd.to_datetime(ts['date'])
    ts = ts.set_index('date').sort_index()
    return ts

term_structure = build_term_structure(futures_all)
print(f'Term structure: {len(term_structure)} trading days')
print(f'Range: {term_structure.index.min().date()} to {term_structure.index.max().date()}')
term_structure.head(10)

## 5. Download VIX Spot & ETF Prices

We pull the VIX index (`^VIX`), VXV (`^VIX3M` since 2018, previously `^VXV`),
and the ETFs: **SVXY**, **VIXY**, **SPY** from Yahoo Finance.

In [ ]:
# Download VIX spot
vix = yf.download('^VIX', start='2006-01-01', end=FULL_END, auto_adjust=True, progress=False)
vix = vix[['Close']].rename(columns={'Close': 'VIX'})
# Flatten any multi-level columns from newer yfinance versions
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)
vix.index = pd.to_datetime(vix.index)
vix.index.name = 'date'
# If still MultiIndex, flatten
if hasattr(vix.index, 'levels'):
    vix.index = vix.index.get_level_values(0)
print(f'VIX: {len(vix)} rows, {vix.index.min().date()} to {vix.index.max().date()}')

# Download VXV / VIX3M (for Cooper's Vratio strategy)
# Yahoo changed the ticker from ^VXV to ^VIX3M
vxv = yf.download('^VIX3M', start='2006-01-01', end=FULL_END, auto_adjust=True, progress=False)
if len(vxv) == 0:
    vxv = yf.download('^VXV', start='2006-01-01', end=FULL_END, auto_adjust=True, progress=False)
vxv = vxv[['Close']].rename(columns={'Close': 'VXV'})
if isinstance(vxv.columns, pd.MultiIndex):
    vxv.columns = vxv.columns.get_level_values(0)
vxv.index = pd.to_datetime(vxv.index)
vxv.index.name = 'date'
if hasattr(vxv.index, 'levels'):
    vxv.index = vxv.index.get_level_values(0)
print(f'VXV/VIX3M: {len(vxv)} rows')

# Download ETFs: SVXY, VIXY, SPY
etf_tickers = ['SVXY', 'VIXY', 'SPY']
etfs = {}
for ticker in etf_tickers:
    tmp = yf.download(ticker, start='2006-01-01', end=FULL_END, auto_adjust=True, progress=False)
    tmp = tmp[['Close']].rename(columns={'Close': ticker})
    if isinstance(tmp.columns, pd.MultiIndex):
        tmp.columns = tmp.columns.get_level_values(0)
    tmp.index = pd.to_datetime(tmp.index)
    tmp.index.name = 'date'
    if hasattr(tmp.index, 'levels'):
        tmp.index = tmp.index.get_level_values(0)
    etfs[ticker] = tmp
    print(f'{ticker}: {len(tmp)} rows, {tmp.index.min().date()} to {tmp.index.max().date()}')

# Also download S&P 500 total return for benchmark
spy_data = etfs['SPY']

## 6. Merge All Data

In [ ]:
# Merge term structure with VIX spot
data = term_structure.join(vix, how='inner')

# Merge VXV
data = data.join(vxv, how='left')

# Merge ETFs
for ticker in etf_tickers:
    data = data.join(etfs[ticker], how='left')

# ── Compute trading days to expiry (business days, more accurate) ──
data['F1_tdte'] = data.apply(
    lambda row: np.busday_count(
        row.name.date(), 
        row['F1_expiry'].date()
    ), axis=1
)
# Avoid division by zero
data['F1_tdte'] = data['F1_tdte'].clip(lower=1)

print(f'Merged dataset: {len(data)} rows')
print(f'Range: {data.index.min().date()} to {data.index.max().date()}')
data[['VIX', 'F1_settle', 'F2_settle', 'F1_dte', 'F1_tdte']].head(10)

## 7. Compute the Daily Roll (Buehler Metric)

$$\text{daily\_roll} = \frac{F_{1,\text{settle}} - \text{VIX}_{\text{spot}}}{\text{trading days to settlement of } F_1}$$

When `daily_roll > 0` the market is in **contango** (futures above VIX).
When `daily_roll < 0` the market is in **backwardation** (VIX above futures).

In [ ]:
data['daily_roll'] = (data['F1_settle'] - data['VIX']) / data['F1_tdte']

print('Daily roll statistics:')
print(data['daily_roll'].describe())

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(data.index, data['VIX'], label='VIX Spot', alpha=0.7)
axes[0].plot(data.index, data['F1_settle'], label='Nearest Futures (F1)', alpha=0.7)
axes[0].set_ylabel('Level')
axes[0].legend()
axes[0].set_title('VIX Spot vs Nearest Futures Contract')

axes[1].plot(data.index, data['daily_roll'], color='steelblue', alpha=0.6, lw=0.7)
axes[1].axhline(SHORT_THRESHOLD, color='green', ls='--', lw=1, label=f'Short threshold (+{SHORT_THRESHOLD})')
axes[1].axhline(LONG_THRESHOLD, color='red', ls='--', lw=1, label=f'Long threshold ({LONG_THRESHOLD})')
axes[1].axhline(0, color='black', ls='-', lw=0.5)
axes[1].set_ylabel('Daily Roll (VIX points)')
axes[1].set_title('Daily Roll = (F1 − VIX) / Trading Days to Settlement')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Contango / Backwardation Statistics (cf. Buehler Exhibit 3)

In [ ]:
# Percentage spread: (F - VIX) / VIX
data['pct_spread_F1'] = (data['F1_settle'] - data['VIX']) / data['VIX']
data['pct_spread_F2'] = (data['F2_settle'] - data['VIX']) / data['VIX']

data['contango_F1'] = data['F1_settle'] > data['VIX']

print('=== Percentage Spread from VIX Spot ===')
for col in ['pct_spread_F1', 'pct_spread_F2']:
    s = data[col].dropna()
    label = col.replace('pct_spread_', '')
    print(f'\n{label}:')
    print(f'  Mean:  {s.mean():.4f}')
    print(f'  Max:   {s.max():.4f}')
    print(f'  Min:   {s.min():.4f}')
    print(f'  Std:   {s.std():.4f}')

n_contango = data['contango_F1'].sum()
n_backwardation = (~data['contango_F1']).sum()
print(f'\nDays in contango (F1 > VIX):      {n_contango} ({n_contango/len(data)*100:.1f}%)')
print(f'Days in backwardation (F1 <= VIX): {n_backwardation} ({n_backwardation/len(data)*100:.1f}%)')
print(f'Ratio:                             {n_contango/max(n_backwardation,1):.1f} : 1')

## 9. Buehler Trading Strategy

**Rules** (from the test-period optimisation, Exhibit 5):

| Daily Roll | Position | ETF |
|---|---|---|
| ≥ +0.0197 | Short VIX (sell vol) | Buy SVXY |
| ≤ −0.240 | Long VIX (buy vol) | Buy VIXY |
| In between | Cash | — |

We generate signals from daily-roll computed on the **previous day's close** (next-day execution).

In [ ]:
def buehler_signals(data, short_thresh=SHORT_THRESHOLD, long_thresh=LONG_THRESHOLD):
    """
    Generate Buehler trading signals.
    Returns a Series with values: 'SVXY', 'VIXY', 'CASH'
    Signal on day t is based on data from day t-1 (next-day execution).
    """
    # Use previous day's roll to generate today's signal
    prev_roll = data['daily_roll'].shift(1)
    
    signal = pd.Series('CASH', index=data.index)
    signal[prev_roll >= short_thresh] = 'SVXY'   # short vol
    signal[prev_roll <= long_thresh]  = 'VIXY'   # long vol
    
    return signal

data['buehler_signal'] = buehler_signals(data)

# Show signal distribution
print('=== Buehler Signal Distribution (full sample) ===')
print(data['buehler_signal'].value_counts())
print(f'\nDays short vol (SVXY): {(data["buehler_signal"]=="SVXY").sum()}')
print(f'Days long  vol (VIXY): {(data["buehler_signal"]=="VIXY").sum()}')
print(f'Days cash:             {(data["buehler_signal"]=="CASH").sum()}')

## 10. Cooper Strategies (for comparison)

**Strategy 3 — Vratio (Contango/Backwardation):**
- If 10-day MA of (VXV / VIX) > 1 → go long SVXY (short vol)
- Else → go long VIXY (long vol)

**Strategy 4 — VRP (Volatility Risk Premium):**
- If 5-day MA of (VIX − 10-day historical volatility of S&P 500) > 0 → go long SVXY
- Else → go long VIXY

In [ ]:
# ── Cooper Strategy 3: Vratio ──
data['vratio'] = data['VXV'] / data['VIX']
data['vratio_ma10'] = data['vratio'].rolling(10).mean()

def cooper_vratio_signals(data):
    prev_vratio = data['vratio_ma10'].shift(1)
    signal = pd.Series('VIXY', index=data.index)  # default: long vol
    signal[prev_vratio > 1.0] = 'SVXY'  # short vol when contango
    signal[prev_vratio.isna()] = 'CASH'
    return signal

data['cooper_vratio_signal'] = cooper_vratio_signals(data)

# ── Cooper Strategy 4: VRP ──
# 10-day historical vol of S&P 500 (annualised), then compare to VIX
spy_rets = etfs['SPY'].pct_change()
spy_rets.columns = ['spy_ret'] if not isinstance(spy_rets.columns, pd.MultiIndex) else spy_rets.columns
# Compute 10-day realised vol, annualised
hvol10 = spy_rets.rolling(10).std() * np.sqrt(252) * 100  # in VIX-like units
hvol10.columns = ['HVOL10']
data = data.join(hvol10, how='left')

data['vrp_raw'] = data['VIX'] - data['HVOL10']
data['vrp_ma5'] = data['vrp_raw'].rolling(5).mean()

def cooper_vrp_signals(data):
    prev_vrp = data['vrp_ma5'].shift(1)
    signal = pd.Series('VIXY', index=data.index)
    signal[prev_vrp > 0] = 'SVXY'
    signal[prev_vrp.isna()] = 'CASH'
    return signal

data['cooper_vrp_signal'] = cooper_vrp_signals(data)

print('Cooper Vratio signals (full sample):')
print(data['cooper_vratio_signal'].value_counts())
print('\nCooper VRP signals (full sample):')
print(data['cooper_vrp_signal'].value_counts())

## 11. Backtest Engine

In [ ]:
def backtest_strategy(data, signal_col, start_date, end_date, label='Strategy'):
    """
    Backtest a strategy that switches between SVXY, VIXY, or CASH.
    Returns a DataFrame with daily equity values.
    """
    df = data.loc[start_date:end_date].copy()
    df = df.dropna(subset=['SVXY', 'VIXY'])
    
    if len(df) == 0:
        print(f'WARNING: No data for {label} in {start_date} to {end_date}')
        return pd.DataFrame()
    
    # Compute daily returns for ETFs
    df['SVXY_ret'] = df['SVXY'].pct_change()
    df['VIXY_ret'] = df['VIXY'].pct_change()
    df['SPY_ret']  = df['SPY'].pct_change()
    
    # Strategy return: use the ETF return corresponding to the signal
    df['strat_ret'] = 0.0
    df.loc[df[signal_col] == 'SVXY', 'strat_ret'] = df.loc[df[signal_col] == 'SVXY', 'SVXY_ret']
    df.loc[df[signal_col] == 'VIXY', 'strat_ret'] = df.loc[df[signal_col] == 'VIXY', 'VIXY_ret']
    # CASH → 0 return
    
    # Equity curves
    df['strat_equity'] = (1 + df['strat_ret']).cumprod()
    df['spy_equity']   = (1 + df['SPY_ret']).cumprod()
    
    # Stats
    n_days = len(df)
    n_years = n_days / 252
    total_ret = df['strat_equity'].iloc[-1] - 1
    cagr = (df['strat_equity'].iloc[-1]) ** (1/n_years) - 1 if n_years > 0 else 0
    
    daily_rets = df['strat_ret'].dropna()
    avg_ret = daily_rets.mean() * 252
    vol = daily_rets.std() * np.sqrt(252)
    sharpe = avg_ret / vol if vol > 0 else 0
    
    # Sortino
    downside = daily_rets[daily_rets < 0]
    semi_std = downside.std() * np.sqrt(252)
    sortino = avg_ret / semi_std if semi_std > 0 else 0
    
    # Max drawdown
    cum = df['strat_equity']
    rolling_max = cum.cummax()
    drawdown = (cum - rolling_max) / rolling_max
    max_dd = drawdown.min()
    
    # Number of trades (signal changes)
    n_trades = (df[signal_col] != df[signal_col].shift(1)).sum()
    
    # Days in each position
    days_long  = (df[signal_col] == 'VIXY').sum()
    days_short = (df[signal_col] == 'SVXY').sum()
    days_cash  = (df[signal_col] == 'CASH').sum()
    
    stats = {
        'Strategy': label,
        'Terminal Value ($1)': f'{df["strat_equity"].iloc[-1]:.3f}',
        'CAGR': f'{cagr:.2%}',
        'Avg Daily Ret (ann.)': f'{avg_ret:.2%}',
        'Volatility (ann.)': f'{vol:.2%}',
        'Sharpe': f'{sharpe:.3f}',
        'Sortino': f'{sortino:.3f}',
        'Max Drawdown': f'{max_dd:.2%}',
        'Num Trades': n_trades,
        'Days Long': days_long,
        'Days Short': days_short,
        'Days Cash': days_cash,
        'Days in Market': n_days,
    }
    
    return df, stats

print('Backtest engine ready.')

## 12. Test Period: Buehler Optimisation (Jan 2008 – Dec 2010)

This replicates the **in-sample** optimisation from the paper (originally Jan 2007, but we start
Jan 2008 due to data quality issues in the old-format CBOE archive files for 2006–2007).
We use VIX futures directly (no ETFs existed yet for most of this period).

In [ ]:
# For the test period, we can only use futures (ETFs didn't exist)
# We simulate: short VIX futures → daily P&L from futures roll
# But for simplicity and comparability, we compute the futures-based returns

test_data = data.loc[TEST_START:TEST_END].copy()
test_data['buehler_signal'] = buehler_signals(test_data)

# Approximate futures return as: -(F1 change / F1 prev) for short, +(F1 change / F1 prev) for long
test_data['F1_ret'] = test_data['F1_settle'].pct_change()
test_data['fut_strat_ret'] = 0.0
test_data.loc[test_data['buehler_signal'] == 'SVXY', 'fut_strat_ret'] = -test_data.loc[test_data['buehler_signal'] == 'SVXY', 'F1_ret']
test_data.loc[test_data['buehler_signal'] == 'VIXY', 'fut_strat_ret'] = test_data.loc[test_data['buehler_signal'] == 'VIXY', 'F1_ret']

test_data['fut_equity'] = (1 + test_data['fut_strat_ret']).cumprod()

# Plot test period (cf. Buehler Exhibit 5)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_data.index, test_data['fut_equity'], 'b-', lw=1.5, label='Buehler Futures Strategy')
ax.set_title('VIX Futures Trading Strategy — Test Period (cf. Buehler Exhibit 5)', fontsize=14)
ax.set_ylabel('Value of $1 Investment')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

test_terminal = test_data['fut_equity'].iloc[-1]
test_years = len(test_data) / 252
test_cagr = test_terminal ** (1/test_years) - 1
print(f'\nTest Period Results:')
print(f'  Terminal value: ${test_terminal:.2f}')
print(f'  CAGR:           {test_cagr:.1%}')

## 13. Out-of-Sample Backtest: ETF Strategy (Oct 2011 – Dec 2016)

This is the **core result** matching Buehler Exhibit 6. We compare:
1. **SVXY/VIXY ETF Strategy** (Buehler)
2. **VIX Futures Strategy** (same thresholds, simulated with futures returns)
3. **S&P 500 Buy & Hold**

In [ ]:
# ── Run ETF backtest (Buehler) ──
bt_buehler, stats_buehler = backtest_strategy(
    data, 'buehler_signal', OOS_START, OOS_END, 'SVXY/VIXY Strategy'
)

# ── Simulate futures-only strategy over same period ──
oos_data = data.loc[OOS_START:OOS_END].copy()
oos_data['F1_ret'] = oos_data['F1_settle'].pct_change()
oos_data['fut_strat_ret'] = 0.0
oos_data.loc[oos_data['buehler_signal'] == 'SVXY', 'fut_strat_ret'] = \
    -oos_data.loc[oos_data['buehler_signal'] == 'SVXY', 'F1_ret']
oos_data.loc[oos_data['buehler_signal'] == 'VIXY', 'fut_strat_ret'] = \
    oos_data.loc[oos_data['buehler_signal'] == 'VIXY', 'F1_ret']
oos_data['fut_equity'] = (1 + oos_data['fut_strat_ret']).cumprod()

# ── S&P 500 benchmark ──
spy_oos = data.loc[OOS_START:OOS_END, 'SPY'].dropna()
spy_equity = spy_oos / spy_oos.iloc[0]

# ══════════════════════════════════════════════════════════════════
# MAIN CHART — Buehler Exhibit 6 Replica
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(bt_buehler.index, bt_buehler['strat_equity'], 'k-', lw=2, label='SVXY/VIXY Strategy')
ax.plot(oos_data.index, oos_data['fut_equity'], color='gray', lw=1.5, label='Futures Only')
ax.plot(spy_equity.index, spy_equity, color='lightgray', lw=1.5, label='S&P 500')

ax.set_title('Trading Strategy Results Using VIX ETF and Futures; Long Position in S&P 500 ETF\n'
             f'(Oct 4, 2011 – Dec 30, 2016)', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n=== Buehler Out-of-Sample Results ===')
for k, v in stats_buehler.items():
    print(f'  {k:25s}: {v}')

## 14. Summary Statistics Table (cf. Buehler Exhibit 7)

In [ ]:
# Futures strategy stats
oos_fret = oos_data['fut_strat_ret'].dropna()
fut_terminal = oos_data['fut_equity'].iloc[-1]
fut_years = len(oos_data) / 252
fut_cagr = fut_terminal ** (1/fut_years) - 1
fut_vol = oos_fret.std() * np.sqrt(252)
fut_sharpe = (oos_fret.mean() * 252) / fut_vol if fut_vol > 0 else 0
fut_semi = oos_fret[oos_fret < 0].std() * np.sqrt(252)
fut_sortino = (oos_fret.mean() * 252) / fut_semi if fut_semi > 0 else 0

# S&P 500 stats
spy_ret_oos = data.loc[OOS_START:OOS_END, 'SPY'].pct_change().dropna()
spy_terminal = spy_equity.iloc[-1]
spy_cagr = spy_terminal ** (1/fut_years) - 1
spy_vol = spy_ret_oos.std() * np.sqrt(252)
spy_sharpe = (spy_ret_oos.mean() * 252) / spy_vol if spy_vol > 0 else 0
spy_semi = spy_ret_oos[spy_ret_oos < 0].std() * np.sqrt(252)
spy_sortino = (spy_ret_oos.mean() * 252) / spy_semi if spy_semi > 0 else 0

summary = pd.DataFrame({
    'ETF Strategy': [
        f'{bt_buehler["strat_equity"].iloc[-1]:.3f}',
        stats_buehler['CAGR'],
        stats_buehler['Avg Daily Ret (ann.)'],
        stats_buehler['Volatility (ann.)'],
        stats_buehler['Sharpe'],
        stats_buehler['Sortino'],
        stats_buehler['Max Drawdown'],
        stats_buehler['Num Trades'],
        stats_buehler['Days Long'],
        stats_buehler['Days Short'],
        stats_buehler['Days in Market'],
    ],
    'Futures Strategy': [
        f'{fut_terminal:.3f}',
        f'{fut_cagr:.2%}',
        f'{oos_fret.mean()*252:.2%}',
        f'{fut_vol:.2%}',
        f'{fut_sharpe:.3f}',
        f'{fut_sortino:.3f}',
        f'{(oos_data["fut_equity"] / oos_data["fut_equity"].cummax() - 1).min():.2%}',
        f'{(oos_data["buehler_signal"] != oos_data["buehler_signal"].shift(1)).sum()}',
        f'{(oos_data["buehler_signal"]=="VIXY").sum()}',
        f'{(oos_data["buehler_signal"]=="SVXY").sum()}',
        f'{len(oos_data)}',
    ],
    'S&P 500': [
        f'{spy_terminal:.3f}',
        f'{spy_cagr:.2%}',
        f'{spy_ret_oos.mean()*252:.2%}',
        f'{spy_vol:.2%}',
        f'{spy_sharpe:.3f}',
        f'{spy_sortino:.3f}',
        f'{(spy_equity / spy_equity.cummax() - 1).min():.2%}',
        '1',
        f'{len(spy_ret_oos)}',
        '0',
        f'{len(spy_ret_oos)}',
    ]
}, index=[
    'Terminal Value',
    'CAGR',
    'Avg Return (ann.)',
    'Volatility (ann.)',
    'Sharpe Ratio',
    'Sortino Ratio',
    'Max Drawdown',
    'Num Trades',
    'Days Long',
    'Days Short',
    'Days in Market',
])

print('\n=== Summary of Trading Strategy Results (cf. Buehler Exhibit 7) ===\n')
display(summary)

## 15. Cooper Strategies — Out-of-Sample Comparison

We add Cooper's Vratio and VRP strategies for comparison over the same OOS period.

In [ ]:
# Backtest Cooper strategies
bt_vratio, stats_vratio = backtest_strategy(
    data, 'cooper_vratio_signal', OOS_START, OOS_END, 'Cooper Vratio (S3)'
)
bt_vrp, stats_vrp = backtest_strategy(
    data, 'cooper_vrp_signal', OOS_START, OOS_END, 'Cooper VRP (S4)'
)

# ── Combined chart ──
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(bt_buehler.index, bt_buehler['strat_equity'], 'k-', lw=2, label='Buehler ETF Strategy')
if len(bt_vratio) > 0:
    ax.plot(bt_vratio.index, bt_vratio['strat_equity'], 'b--', lw=1.5, label='Cooper Vratio (S3)')
if len(bt_vrp) > 0:
    ax.plot(bt_vrp.index, bt_vrp['strat_equity'], 'r--', lw=1.5, label='Cooper VRP (S4)')
ax.plot(spy_equity.index, spy_equity, color='lightgray', lw=1.5, label='S&P 500')

ax.set_title('All Strategies Compared (Out-of-Sample, Oct 2011 – Dec 2016)', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Summary
comparison = pd.DataFrame([stats_buehler, stats_vratio, stats_vrp]).set_index('Strategy')
display(comparison)

## 16. Extended Backtest (Oct 2011 – Present)

Extend the backtest beyond Buehler's sample to see how the strategies have performed
through Volmageddon (Feb 2018), COVID (Mar 2020), and beyond.

> **Note:** SVXY changed from −1× to −0.5× leverage after Feb 27, 2018. This structural
> break affects the ETF-based backtest. The futures-based version is unaffected.

In [ ]:
# Extended backtest
ext_end = data.dropna(subset=['SVXY','VIXY']).index.max().strftime('%Y-%m-%d')

bt_ext_buehler, stats_ext_buehler = backtest_strategy(
    data, 'buehler_signal', OOS_START, ext_end, 'Buehler (extended)'
)
bt_ext_vratio, stats_ext_vratio = backtest_strategy(
    data, 'cooper_vratio_signal', OOS_START, ext_end, 'Cooper Vratio (extended)'
)
bt_ext_vrp, stats_ext_vrp = backtest_strategy(
    data, 'cooper_vrp_signal', OOS_START, ext_end, 'Cooper VRP (extended)'
)

spy_ext = data.loc[OOS_START:ext_end, 'SPY'].dropna()
spy_ext_eq = spy_ext / spy_ext.iloc[0]

fig, ax = plt.subplots(figsize=(14, 7))

if len(bt_ext_buehler) > 0:
    ax.plot(bt_ext_buehler.index, bt_ext_buehler['strat_equity'], 'k-', lw=2, label='Buehler')
if len(bt_ext_vratio) > 0:
    ax.plot(bt_ext_vratio.index, bt_ext_vratio['strat_equity'], 'b--', lw=1.5, label='Cooper Vratio')
if len(bt_ext_vrp) > 0:
    ax.plot(bt_ext_vrp.index, bt_ext_vrp['strat_equity'], 'r--', lw=1.5, label='Cooper VRP')
ax.plot(spy_ext_eq.index, spy_ext_eq, color='lightgray', lw=1.5, label='S&P 500')

# Mark key events
for evt_date, evt_label in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt_dt = pd.Timestamp(evt_date)
    if evt_dt >= bt_ext_buehler.index.min() and evt_dt <= bt_ext_buehler.index.max():
        ax.axvline(evt_dt, color='orange', ls=':', alpha=0.6)
        ax.text(evt_dt, ax.get_ylim()[1]*0.9, f' {evt_label}', fontsize=9, color='orange')

ax.set_title(f'Extended Backtest (Oct 2011 – {ext_end})', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Extended stats
ext_comparison = pd.DataFrame([stats_ext_buehler, stats_ext_vratio, stats_ext_vrp]).set_index('Strategy')
display(ext_comparison)

## 17. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, (bt, label, color) in zip(axes, [
    (bt_ext_buehler, 'Buehler', 'black'),
    (bt_ext_vratio, 'Cooper Vratio', 'blue'),
    (bt_ext_vrp, 'Cooper VRP', 'red'),
]):
    if len(bt) == 0:
        continue
    eq = bt['strat_equity']
    dd = (eq / eq.cummax()) - 1
    ax.fill_between(dd.index, dd.values, 0, color=color, alpha=0.3)
    ax.plot(dd.index, dd.values, color=color, lw=0.8)
    ax.set_ylabel('Drawdown')
    ax.set_title(f'{label} — Max Drawdown: {dd.min():.1%}')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 18. Threshold Sensitivity Analysis

How sensitive is the Buehler strategy to the choice of thresholds?
We scan a grid around the published values.

In [ ]:
short_thresholds = np.arange(0.00, 0.08, 0.005)
long_thresholds  = np.arange(-0.40, -0.05, 0.02)

results = []
for st in short_thresholds:
    for lt in long_thresholds:
        data['_tmp_signal'] = buehler_signals(data, short_thresh=st, long_thresh=lt)
        try:
            bt_tmp, stats_tmp = backtest_strategy(
                data, '_tmp_signal', OOS_START, OOS_END, f'ST={st:.3f},LT={lt:.3f}'
            )
            if len(bt_tmp) > 0:
                results.append({
                    'short_thresh': st,
                    'long_thresh': lt,
                    'terminal': bt_tmp['strat_equity'].iloc[-1],
                    'sharpe': float(stats_tmp['Sharpe']),
                    'max_dd': float(stats_tmp['Max Drawdown'].strip('%')) / 100,
                })
        except:
            pass

if 'results' in dir() and len(results) > 0:
    res_df = pd.DataFrame(results)
    
    # Heatmap of terminal value
    pivot = res_df.pivot_table(values='terminal', index='long_thresh', columns='short_thresh')
    
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values, cmap='RdYlGn', shading='auto')
    ax.set_xlabel('Short Threshold (contango → sell vol)')
    ax.set_ylabel('Long Threshold (backwardation → buy vol)')
    ax.set_title('Terminal Value ($1 invested) — Threshold Sensitivity\n'
                 f'Buehler thresholds marked with ★ (OOS: {OOS_START} to {OOS_END})')
    ax.plot(SHORT_THRESHOLD, LONG_THRESHOLD, 'k*', markersize=15)
    plt.colorbar(im, label='Terminal Value ($)')
    plt.tight_layout()
    plt.show()

# Cleanup
if '_tmp_signal' in data.columns:
    data.drop('_tmp_signal', axis=1, inplace=True)

## Notes & Caveats

1. **Pre-2008 data quality**: The old-format CBOE archive files (2006–2007) have sparse
   trading data and approximate expiration dates, causing erratic term-structure readings.
   We therefore start the analysis from January 2008.

2. **SVXY leverage change**: After the Volmageddon event on Feb 5, 2018, ProShares changed
   SVXY from −1× to −0.5× daily exposure. The extended backtest beyond 2018 uses the
   actual (reduced-leverage) SVXY prices.

3. **Transaction costs** are not included. Buehler also omits them. In practice, bid-ask spreads
   on SVXY/VIXY and commissions will reduce returns.

4. **Signal timing**: We use previous-day-close data to generate next-day signals, matching
   Buehler's methodology.

5. **VIX futures expiration dates** for the old-format files (2006–2012) are approximated as
   the 3rd Wednesday of the contract month. For exact dates, consult the CBOE calendar.

6. **Cooper's strategies** use VXV (now ^VIX3M) which has a shorter history than VIX futures.
   Vratio signals before VXV data begins will show as CASH.